# Extract all the relevant data of all the scans

Wrestle with all the log files of all the scans.
This is both used for double-checking all the scanning and reconstruction parameters to look for inconsistencies to be corrected.
At the end we generate some helping files which we need for collaboration.
And pull some data out to add to [the manuscript we write for this project](https://github.com/habi/sticklebacks-manuscript).

First set up the notebook with some imports and defaults.

In [ ]:
# Load the python modules we need
import platform
import os
import glob
import pandas
import imageio
import numpy
import matplotlib.pyplot as plt
from matplotlib_scalebar.scalebar import ScaleBar
import seaborn
import dask
import dask_image.imread
from dask.distributed import Client, LocalCluster
import skimage
from tqdm import notebook

In [ ]:
# Load our own log file parsing code
# This is loaded as a submodule to alleviate excessive copy-pasting between *all* projects we do
# See https://github.com/habi/BrukerSkyScanLogfileRuminator for details on its inner workings
import BrukerSkyScanLogfileRuminator.parsing_functions as logparse

In [ ]:
# Set dask temporary folder
# Do this before creating a client: https://stackoverflow.com/a/62804525/323100
import tempfile
if 'Linux' in platform.system():
    # Check if me mounted the FastSSD, otherwise go to standard tmp file
    if os.path.exists(os.path.join(os.sep, 'media', 'habi', 'Fast_SSD')):
        tmp = os.path.join(os.sep, 'media', 'habi', 'Fast_SSD', 'tmp')
    else:
        tmp = tempfile.gettempdir()
elif 'Darwin' in platform.system():
    tmp = tempfile.gettempdir()
else:
    if 'anaklin' in platform.node():
        tmp = os.path.join('F:\\tmp')
    else:
        tmp = os.path.join('D:\\tmp')
dask.config.set({'temporary_directory': tmp})
print('Dask temporary files go to %s' % dask.config.get('temporary_directory'))

In [ ]:
from dask.distributed import Client
client = Client()

In [ ]:
client

In [ ]:
print('You can see what DASK is doing at "http://localhost:%s/status"' % client.scheduler_info()['services']['dashboard'])

In [ ]:
# Set up figure defaults
plt.rc('image', cmap='gray', interpolation='nearest')  # Display all images in b&w and with 'nearest' interpolation
plt.rcParams['figure.figsize'] = (16, 9)  # Size up figures a bit
plt.rcParams['figure.dpi'] = 200

In [ ]:
# Setup scale bar defaults
plt.rcParams['scalebar.location'] = 'lower right'
plt.rcParams['scalebar.frameon'] = False
plt.rcParams['scalebar.color'] = 'white'

Since the (tomographic) data can reside on different drives we set a folder to use below

In [ ]:
local = True
if local:
    # Load the log files from the repository subfolder.
    # Then we cannot 
    Root = os.path.join(os.getcwd(), 'logfiles')
else:
    Root = os.path.join('/home/habi/research_storage_ben/microCT_Stickleback/')
# Root = os.path.join('/media/habi/Fast_SSD/IEE Stickleback/')
print('We are loading all the data from %s' % Root)

Now that we are set up, actually start to load/ingest the data.

In [ ]:
# Make us a dataframe for saving all that we need
Data = pandas.DataFrame()

In [ ]:
# Get *all* log files, unsorted but fast
Data['LogFile'] = [os.path.join(root, name)
                   for root, dirs, files in os.walk(Root)
                   for name in files
                   if name.endswith((".log"))]

The notebook might not be running locally on our machines, but on Binder.
There, the user has no access to the log files, so we fail back to a local copy of them.
This also means that no reconstructions are available, und we thus cannot count them.
We thus set a variable which skips looking for parameters related to the reconstructions.

In [ ]:
if not len(Data):
    # Our dataframe is empty.
    # We might be running on Binder, e.g. load the logfiles from the subfolder in this repository
    print(10 * ' -', 'CAVEAT', 10 * ' -')
    print('You are most probably running the notebook on binder.')
    print('And thus do not have access to the log files on the research storage')
    print('We are using a "local" copy of the data in the `logfiles` subfolder')
    print('This gives correct, but possibly outdated results...')
    print(10 * ' -', 'CAVEAT', 10 * ' -')
    # Change root folder
    Root = 'logfiles'
    # Load log files again
    Data['LogFile'] = [f for f in sorted(glob.glob(os.path.join(Root, '**', '*.log'),
                                                   recursive=True),
                                         key=os.path.getmtime)]
    running_on_binder = True
else:
    running_on_binder = False

In [ ]:
# Get all folders
Data['Folder'] = [os.path.dirname(f) for f in Data['LogFile']]
Data['FolderShort'] = [f[len(Root)+1:] for f in Data['Folder']]

In [ ]:
if not running_on_binder:
    # Check for samples which are not yet reconstructed
    for c, row in Data.iterrows():
        # Iterate over every 'proj' folder
        if 'proj' in row.Folder:
            if 'TScopy' not in row.Folder and 'PR' not in row.Folder:
                # If there's nothing with 'rec*' on the same level, then tell us
                if not glob.glob(row.Folder.replace('proj', 'rec')):
                    if 'proj2' not in row.LogFile:  # exclude failed scans for Sticklebucket_14 and _15
                        print('- %s is missing matching reconstructions' % row.LogFile[len(Root) + 1:])

In [ ]:
# Get rid of all the logfiles from all the folders that might be on disk but that we don't want to load the data from
for c, row in Data.iterrows():
    if 'ucket' not in row.Folder:  # Only use the scans named Bucket* here, e.g. BucketOfFish_* and Sticklebucket_*
        Data.drop([c], inplace=True)
    elif 'rec' not in row.Folder:  # Only look at logs in the rec folders
        Data.drop([c], inplace=True)
    elif '_regions' in row.Folder:  # Exclude all log files that we write in this notebook (to $scan$_region folders)
        Data.drop([c], inplace=True)
    elif os.path.split(row.LogFile)[1].startswith('._'):  # Remove macos metadata files for files on external storage
        Data.drop([c], inplace=True)
    elif 'BucketOfFish_A' in row.LogFile and '15um_rec' in row.LogFile:  # BucketOfFish_A is the first bucket. For that one we did some test scans. Only use 17.5_ 
        Data.drop([c], inplace=True)
    elif 'BucketOfFish_A' in row.LogFile and '18um_rec' in row.LogFile:  # BucketOfFish_A is the first bucket. For that one we did some test scans. Only use 17.5_ 
        Data.drop([c], inplace=True)
    elif 'BucketOfFish_A' in row.LogFile and '19um_rec' in row.LogFile:  # BucketOfFish_A is the first bucket. For that one we did some test scans. Only use 17.5_ 
        Data.drop([c], inplace=True)
    elif 'Sticklebucket_14' in row.LogFile and 'rec3' in row.LogFile:  # Sticklebucket_14/rec3 is a repeat scan of Sticklebucket_14/rec. Sheila and Ben took all the data for the manuscript from the rec scan though, hence do not look at the rec3 scan here
        Data.drop([c], inplace=True)        
    elif 'Sticklebucket_15' in row.LogFile and 'rec3' in row.LogFile:  # Sticklebucket_14/rec3 is a repeat scan of Sticklebucket_14/rec. Sheila and Ben took all the data for the manuscript from the rec scan though, hence do not look at the rec3 scan here
        Data.drop([c], inplace=True)                
# Reset dataframe to something that we would get if we only would have loaded the 'rec' files
Data = Data.reset_index(drop=True)

In [ ]:
Data.head()

In [ ]:
# Generate us some meaningful colums
Data['Bucket'] = [l[len(Root) + 1:].split(os.sep)[0] for l in Data['LogFile']]
Data['Scan'] = ['.'.join(l[len(Root) + 1:].split(os.sep)[1:-1]) for l in Data['LogFile']]

In [ ]:
Data.head()

In [ ]:
# Get parameters related to scan from logfiles
Data['Scan date'] = [logparse.scandate(log) for log in Data['LogFile']]
Data['Scanner'] = [logparse.scanner(log) for log in Data['LogFile']]
Data['Voltage'] = [logparse.voltage(log) for log in Data['LogFile']]
Data['Current'] = [logparse.current(log) for log in Data['LogFile']]
Data['Filter'] = [logparse.whichfilter(log) for log in Data['LogFile']]
Data['Exposuretime'] = [logparse.exposuretime(log) for log in Data['LogFile']]
Data['Averaging'] = [logparse.averaging(log) for log in Data['LogFile']]
Data['Number of projections'] = [logparse.numproj(log) for log in Data['LogFile']]
Data['ProjectionSize'] = [logparse.projection_size(log) for log in Data['LogFile']]
Data['RotationStep'] = [logparse.rotationstep(log) for log in Data['LogFile']]
Data['ThreeSixty'] = [logparse.threesixtyscan(log) for log in Data['LogFile']]
Data['Voxelsize'] = [logparse.pixelsize(log) for log in Data['LogFile']]
Data['Duration'] = [logparse.duration(log) for log in Data['LogFile']]
Data['Stacks'] = [logparse.stacks(log) for log in Data['LogFile']]

In [ ]:
# Get parameters related to reconstruction from logfiles
Data['Number of reconstructions'] = [logparse.slice_number(log) for log in Data['LogFile']]  # This is the number of reconstructions that NRecon wrote to disk and to the log file. *Not* necessarily the number that may be present on disk. In the InMice project we check if files on disk are the same as filenumber written to log file. But for this manuscript only reading from the log is good enough.
Data['ReconstructionSize'] = [logparse.reconstruction_size(log) for log in Data['LogFile']]
Data['Grayvalue'] = [logparse.reconstruction_grayvalue(log) for log in Data['LogFile']]
Data['RingartefactCorrection'] = [logparse.ringremoval(log) for log in Data['LogFile']]
Data['BeamHardeningCorrection'] = [logparse.beamhardening(log) for log in Data['LogFile']]
Data['ROI'] = [logparse.region_of_interest(log) for log in Data['LogFile']]
Data['NRecon'] = [logparse.nreconversion(log)[1] for log in Data['LogFile']]

In [ ]:
# Search for Fish IDs in `rec_regions`, to show how many *fish* we actually scanned
# In the BucketSeparator-Script we wrote out log files for each extraction
# Above, we dropped all these log files, so we have to construct the folder name again :)
Data['FishRegionsFolder'] = None
for c, row in Data.iterrows():
    # Search a folder up from row.Folder, which is the 'rec' folder
    for root, dirs, files in os.walk(os.path.dirname(row.Folder)):
        # "os.walk returns a 3-tuple containing the directory path, a list of sub-directories, and a list of file names for each directory it visits"
        # So let's look at the 'dirs' that end with 'rec_regions'
        # Then we save that folder into the dataframe
        for directory in dirs:
            if directory.endswith('rec_regions'):
                Data.at[c, 'FishRegionsFolder'] = os.path.join(os.path.dirname(row.Folder), directory)

In [ ]:
# Now that we have the correct folder, we can 'extract' the Fish IDs by simply listing the folders
# These FishIDs are **NOT** in the correct order here!
# But since this notebook is only for reporting in the manuscript, this doesn't matter here.
Data['FishIDs'] = None
for c, row in Data.iterrows():
    # Here, a simple listdir is sufficient to pull all the folder names
    # Find only folders that are "??.X??.???"
    import re
    # Thanks ChatGPT for the regex help!
    Data.at[c, 'FishIDs'] = [f for f in sorted(os.listdir(row.FishRegionsFolder)) if os.path.isdir(os.path.join(row.FishRegionsFolder, f)) and re.fullmatch(r'[A-Z]{2}\.X\d{2}\.\d{3}', f)] 

In [ ]:
Data.head()

In [ ]:
# Sort dataframe by scan date
Data.sort_values(by=['Scan date'], inplace=True)
# Reset dataframe index
Data = Data.reset_index(drop=True)

In [ ]:
# Quickly look at all the different values in our dataframe
DoNotWant = {'LogFile',
             'Folder',
             'FolderShort',
             'Bucket',
             'Scan date'}
for column in Data.columns:
    if column not in DoNotWant:
        print(80*'-')
        print(column, numpy.sort(Data[column].explode().unique()))

In [ ]:
Data.head()

Now that we have "everything" we want in this dataframe, we prepare it for reporting.
According to Ben, "[w]e also excluded Wik lake from all samples (as it had n =1)".
We remove this FishID from our dataframe with a small helper function.

In [ ]:
def remove_wk_ids(item):
    '''Helper function to filter out "WK." IDs from a list/string list'''
    # Convert string representation of a list back to a Python list if necessary
    fish_list = ast.literal_eval(item) if isinstance(item, str) else item   
    if not isinstance(fish_list, list):
        return []
    # Keep only the IDs that do NOT start with "WK."
    return [fish_id for fish_id in fish_list if not fish_id.startswith("WK.")]
# Apply the cleaning function to the DataFrame column permanently
Data["FishIDs"] = Data["FishIDs"].apply(remove_wk_ids)

Numbers for abstract and materials & methods

In [ ]:
# How many scans did we do?
print('We performed %s scans in total' % len(Data.Scan))

In [ ]:
# How many fish did we actually scan?
NumFishTotal = len(Data['FishIDs'].explode().unique())
print('We scanned N=%s fish (unique Fish IDs)' % (NumFishTotal))

In [ ]:
# How many fish per lake did we scan?
# https://github.com/habi/sticklebacks-manuscript/issues/28
Lakes = [fish_id.split('.')[0] for fish_id in Data['FishIDs'].explode() if pandas.notnull(fish_id)]

In [ ]:
# We could from directly count the occurrences with .value_counts() in pandas
# But using 'Counter' makes it more explicit to map the abbreviation to the lake name
from collections import Counter

# Names are manually copied from CT_Sticklebacks_Tracking_Sheet.xlsx and from Bens input
lake_names = {
    "FG": "Finger lake",
    "SL": "Spirit lake",
    "SR": "South Rolly lake",
    "WT": "Watson lake",
    "TL": "Tern lake",
    "WB": "Walby lake",
}

LakeCounts = Counter(
    lake_names.get(lake, lake)
    for lake in Lakes
)

# .most_common() returns a list of (key, value) tuples sorted by count descending
sorted_counts = LakeCounts.most_common()

print(f'We scanned fish from {len(LakeCounts)} lakes:')
print('The total number of fish for each lake are:')
print(', '.join(f'{lake} ({count})' for lake, count in sorted_counts))
print(f'for a total of {NumFishTotal} specimens')

Numbers for micro-CT imaging section in materials and methods

In [ ]:
for i in Data.keys():
    print(i)

In [ ]:
print(f'The X-ray source was set to a voltage of {Data.Voltage.unique()} kV and a current of around {Data.Current.mean()} µA for all but one scan where we used a source voltage of 49 kV and 159 µA due to operator error.')

In [ ]:
Data.Filter.unique()

In [ ]:
print(f'For each sample, we recorded a set of {Data['Number of projections'].unique()} projections of approximately {Data.ProjectionSize.unique()} pixels at {Data.RotationStep.unique()}° intervals over a 360° sample rotation.')

In [ ]:
Data.ThreeSixty.unique()

In [ ]:
print(f'Every single projection was exposed for {round(Data.Exposuretime.mean())} ms.')

In [ ]:
print(f'Because of the length of the fish, we had to acquire so-called stacked scans, on average we scanned {round(Data.Stacks.mean(), 2)} fields of view along the rotation axis of the sample holder.')

In [ ]:
Data['Total Duration'] = [st * stk for st, stk in zip(Data['Duration'], Data['Stacks'])]

In [ ]:
print(f'This resulted in an average scan time of {pandas.to_timedelta(Data['Total Duration'].sum() / len(Data), unit='s')} for each scan.')

In [ ]:
print(f'The projection images were then subsequently reconstructed into stacks of 8bit PNG images with NRecon (Bruker microCT, Kontich, Belgium. Version: {Data.NRecon.unique()}), without applying any ring artefact or beam hardening correction.')

In [ ]:
Data.BeamHardeningCorrection.unique()

In [ ]:
Data.RingartefactCorrection.unique()

In [ ]:
print(f'The isometric voxel sizes in the resulting datasets vary from {sorted(Data.Voxelsize.unique())} µm.')

Add to results

In [ ]:
print(f'A total of {NumFishTotal} unique specimens were scanned in {len(Data)} different scans with a total scanning duration of nearly {pandas.to_timedelta(Data['Total Duration'].sum(), unit='s')}.')
#We acquired 136838 projections, reconstructed into a total of 154622 reconstructions, resulting in approximately 4069 files per scan (N=38).

In [ ]:
print(f'We acquired {Data['Number of projections'].sum()} projections,')
print(f'reconstructed into a total of {Data['Number of reconstructions'].sum()} reconstructions, ')
print(f'resulting in approximately {round(Data['Number of reconstructions'].sum() / len(Data))} files per scan (N={len(Data)}).')

Table and table caption

In [ ]:
print(f'This is {pandas.to_timedelta(Data['Total Duration'].sum() / NumFishTotal, unit='s')} per fish.')

In [ ]:
print(f'On average, we scanned {round(NumFishTotal / len(Data), 2)} fish per scan for a total scan time of {pandas.to_timedelta(Data['Total Duration'].sum(), unit='s')}.')

In [ ]:
# Save 'data' file for https://github.com/habi/sticklebacks-manuscript
# Since the manuscript data is in a subfolder of this notebook on the local machine, we can simply write the output there
if not running_on_binder:
    Data[['FolderShort', 'Bucket', 'Scan', 'Scanner', 'Scan date',
          'Voxelsize', 'Voltage', 'Current',
          'Filter', 'Exposuretime', 'Averaging',
          'Number of projections', 'ProjectionSize', 'RotationStep', 'ThreeSixty',
          'Duration', 'Stacks', 'Total Duration',
          'Number of reconstructions', 'ReconstructionSize', 'ROI',
          'RingartefactCorrection', 'BeamHardeningCorrection', 'Grayvalue', 'NRecon', 'FishIDs'
          ]].to_csv(os.path.join('manuscript', 'content', 'data', 'ScanningDetails.csv'),
                    index=False,
                    header=['Folder', 'Bucket', 'Scan', 'Scanner', 'Scan date',
                            'Voxelsize [μm]', 'Source voltage [kV]', 'Source current [μA]',
                            'Filter', 'Exposure time [ms]', 'Frame averaging',
                            'Number of projections', 'Projection size [px]', 'Rotation step [°]', '360° scan',
                            'Scan duration [s]', 'Stacked scans', 'Total scan duration [s]',                                     
                            'Number of reconstructions', 'Reconstruction size [px]', 'Region of interest for reconstruction',
                            'Ring removal correction', 'Beam hardening correction', 'Gray value mapping', 'NRecon version', 'Fish IDs'
                            ])
print('Saved CSV file with all relevant scanning and reconstruction parameters to',
      os.path.join('manuscript', 'content', 'data', 'ScanningDetails.csv'),
      'for using as supplementary material in the manuscript')